In [1]:
import os 
import pickle
import numpy as np
from qkeras.utils import load_qmodel
#from keras.models import load_model
import hls4ml
from sklearn.metrics import accuracy_score
from hls4ml.model.profiling import numerical
import hls4ml.report as report


2026-10-21 21:36:38.529648: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-21 21:36:38.540402: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 21:36:38.655180: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 21:36:38.655964: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-21 21:36:39.090075: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [2]:
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True


def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))

TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"

X_train_all, y_train_all = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

X_train_all = X_train_all.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

val_idx = int(0.75 * len(X_train_all))
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

n_classes = 6
input_shape = X_train.shape[1:]
train_size = len(X_train)

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...


In [3]:
bits= 4
BASE_DIR = f't1_{bits}'
MODEL_NAME = 'model' 
FINAL_PATH = f"models/{BASE_DIR}/{MODEL_NAME}.h5"

qmodel = load_qmodel(FINAL_PATH)
#qmodel.summary()

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer HeNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


In [ ]:
REUSE_FACTOR  = 1
STRATEGY = 'Resource' # 'Latency', 'Resource'
IO_TYPE = "io_stream" # "io_parallel", 'io_stream'
CLOCK_PERIOD_NS = 10

if CLOCK_PERIOD_NS != 10:
    OUTPUT_DIR = f"/project_hlsm/prj_{BASE_DIR}_{IO_TYPE}_rf{REUSE_FACTOR}_{STRATEGY}_cp{CLOCK_PERIOD_NS}"
else:
    OUTPUT_DIR = f"/project_hlsm/prj_{BASE_DIR}_{IO_TYPE}_rf{REUSE_FACTOR}_{STRATEGY}_fifo100k_100BRAM"
FPGA_PART = "xc7z020clg400-1"  
BACKEND = "Vitis"

hls_config = hls4ml.utils.config_from_keras_model(
    qmodel,
    granularity="name",
    backend='Vitis',
    default_reuse_factor=REUSE_FACTOR,
)

hls_config['LayerName']['output_dense']['Precision']['result'] = 'fixed<16,6,RND,SAT>'
hls_config['Model']['Strategy'] = STRATEGY

# FIFO Buffer Depth Optimization
#  just work with io_stream type
hls_config['Flows'] = ['vitis:fifo_depth_optimization']
hls4ml.model.optimizer.get_optimizer('vitis:fifo_depth_optimization').configure(profiling_fifo_depth=100_000)

# Loading weights from external BRAM
# Layers that have more than 100 weights will be loaded from external BRAM
#hls_config["Model"]["BramFactor"] = 1000

trace_layers(hls_config)
print_dict(hls_config)

hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    output_dir=OUTPUT_DIR,
    part=FPGA_PART,
    clock_period=CLOCK_PERIOD_NS,
    io_type=IO_TYPE,
    backend=BACKEND,
)
hls_model.compile()

Model
  Precision
    default:         fixed<16,6>
  ReuseFactor:       1
  Strategy:          Resource
  BramFactor:        100
  TraceOutput:       True
LayerName
  input_3
    Trace:           True
    Precision
      result:        auto
  conv_0
    Trace:           True
    Precision
      result:        auto
      weight:        fixed<4,1,TRN,WRAP,0>
      bias:          auto
      accum:         auto
    ReuseFactor:     1
    ParallelizationFactor:1
    ConvImplementation:LineBuffer
  conv_0_linear
    Trace:           True
    Precision
      result:        auto
      table:         fixed<18,8,TRN,WRAP,0>
    ReuseFactor:     1
    TableSize:       1024
  pool_0
    Trace:           True
    Precision
      result:        auto
      accum:         auto
    ReuseFactor:     1
    ConvImplementation:LineBuffer
  bn_conv_0
    Trace:           True
    Precision
      result:        auto
      scale:         auto
      bias:          auto
    ReuseFactor:     1
  conv_act_0
    T

In [5]:
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

In [6]:
#numerical(model=qmodel, hls_model=hls_model, X=X_test[:2000], plot='boxplot')

In [7]:
X_test_reduced = np.ascontiguousarray(X_test[:3000])
Y_test_reduced = np.array(y_test[:3000])

y_qkeras = qmodel.predict(X_test_reduced)
y_pred_hls = hls_model.predict(X_test_reduced)

accuracy_qkeras = accuracy_score(
    np.argmax(Y_test_reduced, axis=1),
    np.argmax(y_qkeras, axis=1)
)

accuracy_hls = accuracy_score(
    np.argmax(Y_test_reduced, axis=1),
    np.argmax(y_pred_hls, axis=1)
)

print('Accuracy base: {}'.format(accuracy_qkeras))
print('Accuracy after hls4ml: {}'.format(accuracy_hls))

accuracy_file = os.path.join(OUTPUT_DIR, 'accuracy.txt')

with open(accuracy_file, 'w') as f:
    f.write(f'Accuracy base: {accuracy_qkeras:.6f}\n')
    f.write(f'Accuracy after hls4ml: {accuracy_hls:.6f}\n')

94/94 [==============================] - 0s 1ms/step
Accuracy base: 0.946
Accuracy after hls4ml: 0.945


In [ ]:
hls_model.build(csim=True, synth=True, vsynth=True, cosim=True)


****** vitis-run v2024.2 (64-bit)
  **** SW Build 5239630 on 2024-11-10-11:19:46
  **** Start of session at: Sun Oct 18 00:02:04 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2024.2 5238294
Sourcing Tcl script '/home/fauri/Documentos/hls4ml/sat_6_tensorflow/project_hlsm/prj_paper_4_io_stream_rf1_Resource_fifo100k_100BRAM/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
INFO: [HLS 200-10] Opening project '/home/fauri/Documentos/hls4ml/sat_6_tensorflow/project_hlsm/prj_paper_4_io_stream_rf1_Resource_fifo100k_100BRAM/myproject_prj'.
INFO: [HLS 200-1510] Running: set_top myproject 
INFO: [HLS 200-1510] Running: add_files firmware/myproject.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' to the project
INFO: [HLS 200-1510] Running: add_files -tb myproject_test.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding test

In [ ]:
report.read_vivado_report(hls_model.config.get_output_dir())